In [47]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [48]:
odr = pd.read_csv("data/age-dependency-ratio-old.csv")
tfp = pd.read_csv("data/total-factor-productivity.csv") 
schooling = pd.read_csv("data/average-years-of-schooling-among-adults.csv")
hc = pd.read_csv("data/HC.csv")

# Gør HC fra wide til long format
hc_long = hc.melt(
    id_vars=["ISO code", "Country", "Variable code", "Variable name"],
    var_name="year",
    value_name="hc"
)

# Omdøb kolonner så de matcher dine andre datasæt
hc_long = hc_long.rename(columns={
    "ISO code": "code",
    "Country": "country"
})

# Gør year numerisk
hc_long["year"] = pd.to_numeric(hc_long["year"], errors="coerce")

# Fjern rækker uden år eller HC-værdi
hc_long = hc_long.dropna(subset=["year", "hc"])

# Gør year til integer
hc_long["year"] = hc_long["year"].astype(int)

# Begræns perioden, hvis du vil
hc_long = hc_long[(hc_long["year"] >= 1990) & (hc_long["year"] <= 2020)]

# Behold kun de nødvendige kolonner
hc_long = hc_long[["code", "year", "hc"]]

In [49]:
odr = odr.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "odr"
})

tfp = tfp.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Total factor productivity level": "tfp"
})

schooling = schooling.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Both genders": "schooling"
})

hc_long.head(5)

,code,year,hc
5800,AGO,1990,1.138071
5801,ALB,1990,2.516159
5802,ARE,1990,2.014390
5803,ARG,1990,2.529030
5804,ARM,1990,2.948581


In [ ]:

panel2 = tfp

panel2 = panel2.merge(
    schooling[["code", "year", "schooling"]],
    on=["code", "year"],
    how="inner"
)

panel2 = panel2.merge(
    odr[["code", "year", "odr"]],
    on=["code", "year"],
    how="inner"
)

panel2 = panel2.merge(
    hc_long[["code", "year", "hc"]],
    on=["code", "year"],
    how="inner"
)

panel2["log_tfp"] = np.log(panel2["tfp"])



In [62]:
panel2["ODRxHC"] = panel2["odr"] * panel2["hc"]

In [67]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "tfp ~ odr + hc + ODRxHC + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr", "hc", "ODRxHC"]],
    "std_err": results3.bse[["odr", "hc", "ODRxHC"]],
    "p_value": results3.pvalues[["odr", "hc", "ODRxHC"]],
})

print(main_results3.round(4))

          coef  std_err  p_value
odr    -0.0283   0.0236   0.2293
hc     -0.3710   0.1706   0.0297
ODRxHC  0.0099   0.0067   0.1386
